In [1]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from sklearn.metrics import mean_squared_error, mean_absolute_error


warnings.filterwarnings('ignore')

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [2]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [3]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [4]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [5]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [6]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

In [7]:
df_comparendos_electronicos_copy = df_comparendos_electronicos.copy()
df_comparendos_electronicos_copy['año_mes'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.to_period('M').astype(str)

def preparar_serie_mensual(df, codigo_infraccion):

    infracciones_por_mes = df.groupby(['año_mes', 'COD_INFRACCION'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    
    serie = infracciones_por_mes[infracciones_por_mes['COD_INFRACCION'] == codigo_infraccion].copy()
    serie = serie.set_index('año_mes')['CANTIDAD_INFRACCIONES']
    
    serie.index = pd.to_datetime(serie.index)
    serie = serie.asfreq('MS')
    serie = serie.fillna(0)
    
    return serie

serie_c29 = preparar_serie_mensual(df_comparendos_electronicos_copy, 'C29')
print(f"Serie C29: {len(serie_c29)} meses")
print(f"Desde: {serie_c29.index.min().strftime('%Y-%m')} hasta: {serie_c29.index.max().strftime('%Y-%m')}")
print(f"Total comparendos: {serie_c29.sum():,.0f}")

Serie C29: 96 meses
Desde: 2018-01 hasta: 2025-12
Total comparendos: 284,755


In [8]:
train_start = '2018-01-01'
train_end = '2024-12-01'
test_start = '2025-01-01'
test_end = '2025-12-01'

train_c29 = serie_c29[train_start:train_end].copy()
test_c29 = serie_c29[test_start:test_end].copy()

print(f"Train: {train_c29.index.min().strftime('%Y-%m')} a {train_c29.index.max().strftime('%Y-%m')} ({len(train_c29)} meses)")
print(f"Test: {test_c29.index.min().strftime('%Y-%m')} a {test_c29.index.max().strftime('%Y-%m')} ({len(test_c29)} meses)")

Train: 2018-01 a 2024-12 (84 meses)
Test: 2025-01 a 2025-12 (12 meses)


In [9]:
def time_series_cv_manual(serie, n_ventanas=4, horizonte=12):

    ventanas = []
    
    for i in range(n_ventanas, 0, -1):
        train_end_idx = len(serie) - (i * horizonte)
        train_fold = serie.iloc[:train_end_idx]
        val_fold = serie.iloc[train_end_idx:train_end_idx + horizonte]
        
        ventanas.append((train_fold, val_fold))
        
    return ventanas

ventanas_c29 = time_series_cv_manual(train_c29, n_ventanas=4, horizonte=12)

print("Ventanas de validación generadas:")
for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"Fold {i}: Train {train_fold.index.min().strftime('%Y-%m')} a {train_fold.index.max().strftime('%Y-%m')} "
          f"({len(train_fold)}m) -> Val {val_fold.index.min().strftime('%Y-%m')} a {val_fold.index.max().strftime('%Y-%m')} "
          f"({len(val_fold)}m)")

Ventanas de validación generadas:
Fold 1: Train 2018-01 a 2020-12 (36m) -> Val 2021-01 a 2021-12 (12m)
Fold 2: Train 2018-01 a 2021-12 (48m) -> Val 2022-01 a 2022-12 (12m)
Fold 3: Train 2018-01 a 2022-12 (60m) -> Val 2023-01 a 2023-12 (12m)
Fold 4: Train 2018-01 a 2023-12 (72m) -> Val 2024-01 a 2024-12 (12m)


In [10]:
def evaluar_holt_winters(train_fold, val_fold, estacionalidad=12):

    modelo = ExponentialSmoothing(
        train_fold,
        seasonal_periods=estacionalidad,
        trend='add',
        seasonal='add',
        initialization_method='estimated'
    ).fit()
    
    predicciones = modelo.forecast(len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo

In [11]:
resultados_cv_c29 = []

print("-" * 50)
print("Validación cruzada - Holt-Winters Aditivo (C29)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo = evaluar_holt_winters(train_fold, val_fold)
    
    resultados_cv_c29.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"RMSE: {metricas['RMSE']:.4f}")
    print(f"MAE: {metricas['MAE']:.4f}")
    print(f"MAPE: {metricas['MAPE']:.4f}%")
    print(f"SMAPE: {metricas['SMAPE']:.4f}%")
    print(f"MSE: {metricas['MSE']:.4f}")

metricas_promedio_c29 = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_c29]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_c29]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_c29]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_c29]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_c29])
}

print()
print("-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_c29['RMSE']:.4f}")
print(f"MAE: {metricas_promedio_c29['MAE']:.4f}")
print(f"MAPE: {metricas_promedio_c29['MAPE']:.4f}%")
print(f"SMAPE: {metricas_promedio_c29['SMAPE']:.4f}%")
print(f"MSE: {metricas_promedio_c29['MSE']:.4f}")

--------------------------------------------------
Validación cruzada - Holt-Winters Aditivo (C29)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
RMSE: 869.0270
MAE: 769.3250
MAPE: 29.9843%
SMAPE: 25.0677%
MSE: 755207.8737

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
RMSE: 830.5627
MAE: 676.2801
MAPE: 22.5230%
SMAPE: 23.6492%
MSE: 689834.3365

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
RMSE: 1202.4692
MAE: 961.8149
MAPE: 41.5701%
SMAPE: 28.1845%
MSE: 1445932.1634

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
RMSE: 791.1025
MAE: 587.7869
MAPE: 26.3614%
SMAPE: 21.2314%
MSE: 625843.1901

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 923.2903
MAE: 748.8018
MAPE: 30.1097%
SMAPE: 24.5332%
MSE: 879204.3909


In [12]:
print("-" * 50)
print("Evaluación final en Test (2025)")
print("-" * 50)

modelo_final_c29 = ExponentialSmoothing(
    train_c29,
    seasonal_periods=12,
    trend='add',
    seasonal='add',
    initialization_method='estimated'
).fit()

predicciones_test_c29 = modelo_final_c29.forecast(len(test_c29))

real_test = test_c29.values
pred_test = predicciones_test_c29.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

metricas_test_c29 = {
    'MSE': mse_test,
    'RMSE': rmse_test,
    'MAE': mae_test,
    'MAPE': mape_test,
    'SMAPE': smape_test
}

print(f"RMSE: {rmse_test:.4f}")
print(f"MAE: {mae_test:.4f}")
print(f"MAPE: {mape_test:.4f}%")
print(f"SMAPE: {smape_test:.4f}%")
print(f"MSE: {mse_test:.4f}")

df_resultados_c29 = pd.DataFrame({
    'Métrica': ['RMSE', 'MAE', 'MAPE', 'SMAPE', 'MSE'],
    'CV Promedio': [
        round(metricas_promedio_c29['RMSE'], 4),
        round(metricas_promedio_c29['MAE'], 4),
        round(metricas_promedio_c29['MAPE'], 4),
        round(metricas_promedio_c29['SMAPE'], 4),
        round(metricas_promedio_c29['MSE'], 4)
    ],
    'Test (2025)': [
        round(rmse_test, 4),
        round(mae_test, 4),
        round(mape_test, 4),
        round(smape_test, 4),
        round(mse_test, 4)
    ]
})

print("\nResumen comparativo:")
display(df_resultados_c29)

--------------------------------------------------
Evaluación final en Test (2025)
--------------------------------------------------
RMSE: 378.0217
MAE: 328.1516
MAPE: 28.0029%
SMAPE: 23.0216%
MSE: 142900.3850

Resumen comparativo:


,Métrica,CV Promedio,Test (2025)
0,RMSE,923.2903,378.0217
1,MAE,748.8018,328.1516
2,MAPE,30.1097,28.0029
3,SMAPE,24.5332,23.0216
4,MSE,879204.3909,142900.3850


In [13]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index,
    y=train_c29.values,
    mode='lines+markers',
    name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=test_c29.values,
    mode='lines+markers',
    name='Test Real (2025)',
    line=dict(color='green', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=predicciones_test_c29.values,
    mode='lines+markers',
    name='Predicción Holt-Winters',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2025-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2025-01-01',
    y=1, 
    yref='paper',
    text='<b>Inicio Test</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text=f'C29 - Holt-Winters Aditivo: Predicción vs Real<br>'
             f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()


meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses,
    y=test_c29.values,
    mode='lines+markers',
    name='Real',
    line=dict(color='green', width=2),
    marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses,
    y=predicciones_test_c29.values,
    mode='lines+markers',
    name='Predicción',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Zoom: Test 2025 (Predicho vs Real)',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0,
        y=1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig2.show()


errores_test = test_c29.values - predicciones_test_c29.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses,
    y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple',
    marker_line_color='darkorchid',
    marker_line_width=1,
    opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(
    y=0,
    line_width=1.5,
    line_color='black',
    line_dash='solid',
    opacity=1
)

fig3.update_layout(
    title=dict(
        text='Errores por mes (Real - Predicción) - Test 2025',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig3.show()

In [14]:
def calcular_metricas(real, pred):

    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return metricas

In [15]:
def evaluar_ets(train_fold, val_fold, seasonal_periods=12, config=None):

    if config is None:
        modelo = ETSModel(
            train_fold,
            seasonal_periods=seasonal_periods,
            error='add',
            trend='add',
            seasonal='add',
            damped_trend=True,
            initialization_method='estimated'
        )
    else:
        modelo = ETSModel(
            train_fold,
            seasonal_periods=seasonal_periods,
            error=config.get('error', 'add'),
            trend=config.get('trend', 'add'),
            seasonal=config.get('seasonal', 'add'),
            damped_trend=config.get('damped_trend', False),
            initialization_method='estimated'
        )
    
    modelo_ajustado = modelo.fit(disp=False)
    
    predicciones = modelo_ajustado.forecast(len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    metricas = calcular_metricas(real, pred)
    
    return predicciones, metricas, modelo_ajustado

In [16]:
configuraciones_ets = {
    'ETS_Auto': None,
    'ETS_A_A_A': {
        'error': 'add',
        'trend': 'add', 
        'seasonal': 'add',
        'damped_trend': False
    },
    'ETS_A_Ad_A': {
        'error': 'add',
        'trend': 'add',
        'seasonal': 'add',
        'damped_trend': True
    },
    'ETS_A_A_N': {
        'error': 'add',
        'trend': 'add',
        'seasonal': None,
        'damped_trend': False
    },
    'ETS_A_Ad_N': {
        'error': 'add',
        'trend': 'add',
        'seasonal': None,
        'damped_trend': True
    }
}

In [17]:
resultados_cv_ets = {nombre: [] for nombre in configuraciones_ets.keys()}

print("-" * 50)
print("Validación cruzada - Comparación de modelos ETS (C29)")
print("-" * 50)

for nombre_config, config in configuraciones_ets.items():
    print(f"\n{'─' * 30} {nombre_config} {'─' * 30}")
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c29, 1):
        print(f"\n--- Fold {i} ---")
        
        predicciones, metricas, modelo = evaluar_ets(train_fold, val_fold, config=config)
        
        resultados_cv_ets[nombre_config].append({
            'fold': i,
            'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
            'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
            'metricas': metricas
        })
        
        print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
        print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
        print(f"RMSE: {metricas['RMSE']:.4f}")
        print(f"MAE: {metricas['MAE']:.4f}")
        print(f"MAPE: {metricas['MAPE']:.4f}%")
        print(f"SMAPE: {metricas['SMAPE']:.4f}%")
        print(f"MSE: {metricas['MSE']:.4f}")

metricas_promedio_ets = {}
for nombre_config in configuraciones_ets.keys():
    if len(resultados_cv_ets[nombre_config]) > 0:
        metricas_promedio_ets[nombre_config] = {
            'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_ets[nombre_config]]),
            'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_ets[nombre_config]]),
            'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_ets[nombre_config]]),
            'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_ets[nombre_config]]),
            'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_ets[nombre_config]])
        }

print()
print("-" * 50)
print("Métricas promedio por configuración en validación cruzada")
print("-" * 50)
for nombre_config, metricas in metricas_promedio_ets.items():
    print(f"\n{nombre_config}:")
    print(f"RMSE: {metricas['RMSE']:.4f}")
    print(f"MAE: {metricas['MAE']:.4f}")
    print(f"MAPE: {metricas['MAPE']:.4f}%")
    print(f"SMAPE: {metricas['SMAPE']:.4f}%")
    print(f"MSE: {metricas['MSE']:.4f}")

mejor_config_nombre = min(metricas_promedio_ets, key=lambda k: metricas_promedio_ets[k]['RMSE'])
mejor_config = configuraciones_ets[mejor_config_nombre]
metricas_cv_mejor = metricas_promedio_ets[mejor_config_nombre]

print()
print("-" * 50)
print(f"Mejor configuración: {mejor_config_nombre}")
print("-" * 50)
print(f"RMSE: {metricas_cv_mejor['RMSE']:.4f}")
print(f"MAE: {metricas_cv_mejor['MAE']:.4f}")
print(f"MAPE: {metricas_cv_mejor['MAPE']:.4f}%")
print(f"SMAPE: {metricas_cv_mejor['SMAPE']:.4f}%")
print(f"MSE: {metricas_cv_mejor['MSE']:.4f}")

if mejor_config is None:
    mejor_config = {
        'error': 'add',
        'trend': 'add',
        'seasonal': 'add',
        'damped_trend': True
    }

--------------------------------------------------
Validación cruzada - Comparación de modelos ETS (C29)
--------------------------------------------------

────────────────────────────── ETS_Auto ──────────────────────────────

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
RMSE: 684.9695
MAE: 623.0762
MAPE: 23.9729%
SMAPE: 21.9700%
MSE: 469183.1667

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
RMSE: 764.5283
MAE: 620.7691
MAPE: 22.1981%
SMAPE: 21.3663%
MSE: 584503.5448

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
RMSE: 1169.4003
MAE: 1004.3990
MAPE: 41.6882%
SMAPE: 29.5059%
MSE: 1367496.9640

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
RMSE: 812.9578
MAE: 606.5262
MAPE: 27.2691%
SMAPE: 21.7889%
MSE: 660900.4561

────────────────────────────── ETS_A_A_A ──────────────────────────────

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
RMSE: 572.5594
MAE: 499.1518
MAPE: 18.9282%
SMAPE: 18.2306%
MSE: 327

In [18]:
print("-" * 50)
print("Evaluación final en Test (2025)")
print("-" * 50)

modelo_final_c29 = ETSModel(
    train_c29,
    seasonal_periods=12,
    error=mejor_config.get('error', 'add'),
    trend=mejor_config.get('trend', 'add'),
    seasonal=mejor_config.get('seasonal', 'add'),
    damped_trend=mejor_config.get('damped_trend', False),
    initialization_method='estimated'
).fit(disp=False)

predicciones_test_c29 = modelo_final_c29.forecast(len(test_c29))

real_test = test_c29.values
pred_test = predicciones_test_c29.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

metricas_test_c29 = {
    'MSE': mse_test,
    'RMSE': rmse_test,
    'MAE': mae_test,
    'MAPE': mape_test,
    'SMAPE': smape_test
}

print(f"RMSE: {rmse_test:.4f}")
print(f"MAE: {mae_test:.4f}")
print(f"MAPE: {mape_test:.4f}%")
print(f"SMAPE: {smape_test:.4f}%")
print(f"MSE: {mse_test:.4f}")

df_resultados_c29 = pd.DataFrame({
    'Métrica': ['RMSE', 'MAE', 'MAPE', 'SMAPE', 'MSE'],
    'CV Promedio': [
        round(metricas_cv_mejor['RMSE'], 4),
        round(metricas_cv_mejor['MAE'], 4),
        round(metricas_cv_mejor['MAPE'], 4),
        round(metricas_cv_mejor['SMAPE'], 4),
        round(metricas_cv_mejor['MSE'], 4)
    ],
    'Test (2025)': [
        round(rmse_test, 4),
        round(mae_test, 4),
        round(mape_test, 4),
        round(smape_test, 4),
        round(mse_test, 4)
    ]
})

print("\nResumen comparativo:")
display(df_resultados_c29)

--------------------------------------------------
Evaluación final en Test (2025)
--------------------------------------------------
RMSE: 431.0657
MAE: 363.1522
MAPE: 32.7418%
SMAPE: 25.7811%
MSE: 185817.6157

Resumen comparativo:


,Métrica,CV Promedio,Test (2025)
0,RMSE,774.3795,431.0657
1,MAE,650.2044,363.1522
2,MAPE,25.2321,32.7418
3,SMAPE,22.0126,25.7811
4,MSE,612734.2404,185817.6157


In [19]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c29.index,
    y=train_c29.values,
    mode='lines+markers',
    name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=test_c29.values,
    mode='lines+markers',
    name='Test Real (2025)',
    line=dict(color='green', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c29.index,
    y=predicciones_test_c29.values,
    mode='lines+markers',
    name='Predicción ETS',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2025-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2025-01-01',
    y=1, 
    yref='paper',
    text='<b>Inicio Test</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text=f'C29 - ETS ({mejor_config_nombre}): Predicción vs Real<br>'
             f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()


meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses,
    y=test_c29.values,
    mode='lines+markers',
    name='Real',
    line=dict(color='green', width=2),
    marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses,
    y=predicciones_test_c29.values,
    mode='lines+markers',
    name='Predicción',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Zoom: Test 2025 (Predicho vs Real)',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0,
        y=1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig2.show()


errores_test = test_c29.values - predicciones_test_c29.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses,
    y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple',
    marker_line_color='darkorchid',
    marker_line_width=1,
    opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(
    y=0,
    line_width=1.5,
    line_color='black',
    line_dash='solid',
    opacity=1
)

fig3.update_layout(
    title=dict(
        text='Errores por mes (Real - Predicción) - Test 2025',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig3.show()